## **MLlib: 영화 추천 알고리즘 구현**
+ 데이터셋: MovieLens(2500만개 영화 평점 데이터), 본 예제에서는 7만개의 데이터만 사용 ```ratings_short.csv```
+ 추천 알고리즘: Alternating Least Squares (ALS)

---

### **1. 영화 평점 데이터 불러오기 & 데이터프레임 생성하기**

In [2]:
# [+] SparkSession 설정
from pyspark.sql import SparkSession, Row

spark = (
    SparkSession.builder
    .master("local")
    .appName("recommendation")
    .getOrCreate()
)

sc = spark.sparkContext

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 05:54:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### movielens 데이터 불러오기
+ 파일명: ```ratings_short.csv```
+ 스키마 설정: ```inferSchema=True```
+ 헤더 사용: ```header=True```

In [3]:
# movielens 데이터 불러오기
path = '../data/'
file = 'ratings_short.csv'

ratings_df = spark.read.csv(
    path + file, 
    inferSchema=True, 
    header=True
)

In [4]:
# 데이터프레임 출력
ratings_df.show()

+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|    296|   5.0|1147880044|
|     1|    306|   3.5|1147868817|
|     1|    307|   5.0|1147868828|
|     1|    665|   5.0|1147878820|
|     1|    899|   3.5|1147868510|
|     1|   1088|   4.0|1147868495|
|     1|   1175|   3.5|1147868826|
|     1|   1217|   3.5|1147878326|
|     1|   1237|   5.0|1147868839|
|     1|   1250|   4.0|1147868414|
|     1|   1260|   3.5|1147877857|
|     1|   1653|   4.0|1147868097|
|     1|   2011|   2.5|1147868079|
|     1|   2012|   2.5|1147868068|
|     1|   2068|   2.5|1147869044|
|     1|   2161|   3.5|1147868609|
|     1|   2351|   4.5|1147877957|
|     1|   2573|   4.0|1147878923|
|     1|   2632|   5.0|1147878248|
|     1|   2692|   5.0|1147869100|
+------+-------+------+----------+
only showing top 20 rows



In [5]:
# 데이터프레임 스키마 출력
ratings_df.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)



In [6]:
# [+] 타임스탬프를 제외한 컬럼 선택
ratings_df = ratings_df.select(['userID', 'movieID', 'rating'])

In [7]:
# [+] describe(): 기술 통계량 출력
ratings_df.select('rating').describe().show()

+-------+------------------+
|summary|            rating|
+-------+------------------+
|  count|             71921|
|   mean|3.5821387355570695|
| stddev| 1.042406032579843|
|    min|               0.5|
|    max|               5.0|
+-------+------------------+



### **2. 훈련 데이터 준비 및 영화 추천 모델 학습**

In [8]:
# [+] 훈련 데이터셋과 테스트 데이터셋 분리 (8:2)
train_df, test_df = ratings_df.randomSplit([0.8, 0.2])

In [9]:
# 추천 알고리즘(Alternating Least Squares) 불러오기
from  pyspark.ml.recommendation import ALS

#### **ALS 알고리즘 초매개변수**
+ ```maxIter```: 최대 학습 반복 횟수
+ ```regParam```: 정규화 매개변수(범위: 0~1)
+ ```coldStartStrategy```: 데이터가 부족한 신규 유저 및 아이템에 대한 예측 문제(Cold Start)를 처리하는 방식이며 ```drop```값은 해당 데이터를 모델 학습 과정에서 배제

In [10]:
# 추천 알고리즘 초매개변수 설정
als = ALS(
    maxIter=5,
    regParam=0.1,
    userCol='userID',
    itemCol='movieID',
    ratingCol='rating',
    coldStartStrategy='drop'
)

In [11]:
# [+] 모델 학습
model = als.fit(train_df)

26/11/26 05:55:04 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/11/26 05:55:04 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/11/26 05:55:05 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [12]:
# # 메모리 부족으로 인한 오류 발생시, 아래의 코드를 실행
# from pyspark.sql import SparkSession

# MAX_MEMORY = '5g'
# spark = SparkSession.builder.appName('movie-recommendation')\
#     .config('spark.executor.memory', MAX_MEMORY)\
#     .config('spark.driver.memory', MAX_MEMORY)\
#     .getOrCreate()

In [13]:
# [+] 모델 예측
predictions = model.transform(test_df)

In [14]:
# 예측값 출력
predictions.show()

+------+-------+------+----------+
|userID|movieID|rating|prediction|
+------+-------+------+----------+
|   148|    110|   4.0| 3.7764192|
|   148|    527|   5.0| 4.2793245|
|   148|    589|   4.5| 3.8163455|
|   148|    903|   4.0| 3.9539714|
|   148|    908|   4.0| 4.1343393|
|   148|    913|   4.5| 3.8506243|
|   148|    926|   4.0|  4.149006|
|   148|    955|   3.5| 4.2329745|
|   148|   1080|   3.5| 3.8987014|
|   148|   1136|   4.5| 4.1352043|
|   148|   1201|   5.0| 4.2987094|
|   148|   1208|   5.0| 4.1175213|
|   148|   1234|   4.5| 4.1016665|
|   148|   2329|   4.0| 4.2004857|
|   148|   2571|   5.0|  4.079369|
|   148|   3681|   4.0| 4.0604186|
|   148|   4886|   4.0| 3.8769104|
|   148|   6016|   4.5| 4.2973814|
|   148|  27773|   4.5| 4.3711886|
|   148|  44191|   4.0| 3.9817026|
+------+-------+------+----------+
only showing top 20 rows



In [15]:
# [+] 평점과 예측평점에 대한 통계 출력
predictions.select(['rating', 'prediction']).describe().show()

+-------+------------------+------------------+
|summary|            rating|        prediction|
+-------+------------------+------------------+
|  count|             13714|             13714|
|   mean|3.6075178649555197|3.4231461191502905|
| stddev|1.0403412157870642|0.7544761737521599|
|    min|               0.5|        0.17898998|
|    max|               5.0|          5.704017|
+-------+------------------+------------------+



In [16]:
# 모델 성능 평가: RMSE(Root Mean Squared Error)
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName='rmse',
    labelCol='rating',
    predictionCol='prediction'
)

In [17]:
# [+] RMSE 측정
rmse = evaluator.evaluate(predictions)

In [18]:
rmse

0.915272706852967

### **3. 학습된 모델을 이용한 영화 추천**
+ ```recommendForAllUsers()```: 유저별 아이템 추천
+ ```recommendForAllItems()```: 아이템별 유저 추천
+ ```recommendForUserSubset()```: 특정 유저 그룹에 대한 아이템 추천

In [19]:
# [+] 유저별 아이템을 3개씩 추천
model.recommendForAllUsers(3).show()

+------+--------------------+
|userID|     recommendations|
+------+--------------------+
|     1|[{1209, 5.0394583...|
|     2|[{4833, 5.710121}...|
|     3|[{117364, 5.0471}...|
|     4|[{200818, 4.93746...|
|     5|[{1301, 5.393505}...|
|     6|[{171011, 5.72079...|
|     7|[{142871, 4.75423...|
|     8|[{994, 4.752775},...|
|     9|[{3360, 5.5562286...|
|    10|[{171011, 5.10957...|
|    11|[{1948, 5.1553273...|
|    12|[{7099, 4.8173285...|
|    13|[{142871, 4.98661...|
|    14|[{7099, 5.6016226...|
|    15|[{714, 6.021756},...|
|    16|[{714, 5.5194864}...|
|    17|[{7318, 4.779938}...|
|    18|[{198185, 4.90893...|
|    19|[{141432, 4.94694...|
|    20|[{171011, 5.59181...|
+------+--------------------+
only showing top 20 rows



In [20]:
# [+] 아이템별 유저를 3명씩 추천
model.recommendForAllItems(3).show()

+-------+--------------------+
|movieID|     recommendations|
+-------+--------------------+
|      1|[{327, 5.0318}, {...|
|      2|[{327, 4.713159},...|
|      3|[{210, 4.859692},...|
|      4|[{210, 4.4148846}...|
|      5|[{210, 4.573028},...|
|      6|[{451, 5.325004},...|
|      7|[{22, 4.6584606},...|
|      9|[{210, 4.9840827}...|
|     10|[{451, 4.7812924}...|
|     11|[{333, 5.290491},...|
|     12|[{198, 4.948905},...|
|     13|[{275, 5.113895},...|
|     14|[{451, 4.9320784}...|
|     15|[{400, 4.426473},...|
|     16|[{451, 5.5650735}...|
|     17|[{240, 4.828816},...|
|     18|[{153, 5.314366},...|
|     19|[{327, 4.689189},...|
|     20|[{484, 4.6476326}...|
|     21|[{28, 4.2211027},...|
+-------+--------------------+
only showing top 20 rows



**특정 유저 그룹에 대한 아이템 추천**
1. 해당 유저들의 ID 정보를 데이터프레임으로 생성
2. ```recommendForUserSubset()```를 이용한 아이템 추천

In [21]:
# 특정 유저 ID 집합: 2차원 리스트
user_lst = [[1]]

In [22]:
# 스키마 정의
from pyspark.sql.types import StructType, StructField, IntegerType
schema = StructType([
    StructField('userID', IntegerType(), True)
])

In [23]:
# 데이터프레임 생성
users_df = spark.createDataFrame([(5,)], schema)

In [24]:
users_df.show()

+------+
|userID|
+------+
|     5|
+------+



In [25]:
# [+] recommendForUserSubset(): 특정 유저 그룹에 대한 아이템 추천
user_recs = model.recommendForUserSubset(users_df, 10)

In [26]:
user_recs.show()

+------+--------------------+
|userID|     recommendations|
+------+--------------------+
|     5|[{1301, 5.393505}...|
+------+--------------------+



In [27]:
# 추천 결과를 Python 객체로 받아오기
movies_lst = user_recs.collect()[0].recommendations

In [28]:
movies_lst

[Row(movieID=1301, rating=5.393505096435547),
 Row(movieID=171011, rating=5.3540568351745605),
 Row(movieID=4102, rating=5.2996392250061035),
 Row(movieID=117364, rating=5.220293998718262),
 Row(movieID=68522, rating=5.220293998718262),
 Row(movieID=6286, rating=5.220293998718262),
 Row(movieID=1564, rating=5.197793483734131),
 Row(movieID=1398, rating=5.197793483734131),
 Row(movieID=735, rating=5.156083106994629),
 Row(movieID=8582, rating=5.155691623687744)]

In [29]:
# movies_lst에 대한 데이터프레임 생성
recs_df = spark.createDataFrame(movies_lst)
recs_df.show()

+-------+------------------+
|movieID|            rating|
+-------+------------------+
|   1301| 5.393505096435547|
| 171011|5.3540568351745605|
|   4102|5.2996392250061035|
| 117364| 5.220293998718262|
|  68522| 5.220293998718262|
|   6286| 5.220293998718262|
|   1564| 5.197793483734131|
|   1398| 5.197793483734131|
|    735| 5.156083106994629|
|   8582| 5.155691623687744|
+-------+------------------+



In [31]:
# [+] 영화 데이터에 대한 데이터프레임 생성
file = 'movies.csv'

movies_df = spark.read.csv(
    path + file,
    inferSchema=True,
    header=True
)
movies_df.show()

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
|      6|         Heat (1995)|Action|Crime|Thri...|
|      7|      Sabrina (1995)|      Comedy|Romance|
|      8| Tom and Huck (1995)|  Adventure|Children|
|      9| Sudden Death (1995)|              Action|
|     10|    GoldenEye (1995)|Action|Adventure|...|
|     11|American Presiden...|Comedy|Drama|Romance|
|     12|Dracula: Dead and...|       Comedy|Horror|
|     13|        Balto (1995)|Adventure|Animati...|
|     14|        Nixon (1995)|               Drama|
|     15|Cutthroat Island ...|Action|Adventure|...|
|     16|       Casino (1995)|         Crime|Drama|
|     17|Sen

In [34]:
# [+] recs_df, movies_df 에 대한 Temporary View 생성
recs_df.createOrReplaceTempView('recommendations')
movies_df.createOrReplaceTempView('movies')

In [35]:
# SQL JOIN 연산을 통해 추천된 영화 제목 받아오기
spark.sql("""
    SELECT *
    FROM movies JOIN recommendations
    ON movies.movieID = recommendations.movieID
    ORDER BY rating DESC
""").show()

+-------+--------------------+--------------------+-------+------------------+
|movieId|               title|              genres|movieID|            rating|
+-------+--------------------+--------------------+-------+------------------+
|   1301|Forbidden Planet ...|        Drama|Sci-Fi|   1301| 5.393505096435547|
| 171011|Planet Earth II (...|         Documentary| 171011|5.3540568351745605|
|   4102|Eddie Murphy Raw ...|  Comedy|Documentary|   4102|5.2996392250061035|
| 117364|      Virunga (2014)|     Documentary|War| 117364| 5.220293998718262|
|  68522|        Earth (2007)|         Documentary|  68522| 5.220293998718262|
|   6286|Man Without a Pas...|  Comedy|Crime|Drama|   6286| 5.220293998718262|
|   1564|For Roseanna (Ros...|Comedy|Drama|Romance|   1564| 5.197793483734131|
|   1398|In Love and War (...|         Romance|War|   1398| 5.197793483734131|
|    735|Cemetery Man (Del...|              Horror|    735| 5.156083106994629|
|   8582|Manufacturing Con...|     Documentary|War| 

### **4. 유저 별 영화 추천 서비스를 간단하게 구현하기**
1. SQL문 작성
2. 영화 추천 함수 작성
3. 영화 추천 테스트

In [36]:
# SQL JOIN 연산을 통해 추천된 영화 제목 받아오기
query = """
SELECT * 
FROM movies JOIN recommendations 
ON movies.movieID = recommendations.movieID
ORDER BY rating DESC
"""

In [37]:
# 입력된 유저에 대한 영화 추천 함수
def get_recommendations(user_id, num_recs):
    users_df = spark.createDataFrame([user_id], IntegerType()).toDF('userID')
    users_recs_df = model.recommendForUserSubset(users_df, num_recs)
    recs_lst = users_recs_df.collect()[0].recommendations
    recs_df = spark.createDataFrame(recs_lst)
    recs_df.createOrReplaceTempView('recommendations')
    recommended_movies = spark.sql(query)
    
    return recommended_movies

In [38]:
# 1번 유저에 대한 영화 5개 추천
recs = get_recommendations(1, 5)

In [39]:
# 추천 결과 출력
recs.show()

+-------+--------------------+--------------------+-------+-----------------+
|movieId|               title|              genres|movieID|           rating|
+-------+--------------------+--------------------+-------+-----------------+
|   1209|Once Upon a Time ...|Action|Drama|Western|   1209|5.039458274841309|
|   8827|Bill Cosby, Himse...|  Comedy|Documentary|   8827|5.014345169067383|
|    714|     Dead Man (1995)|Drama|Mystery|Wes...|    714|5.005677223205566|
|   3470|  Dersu Uzala (1975)|     Adventure|Drama|   3470|4.968241214752197|
|   2067|Doctor Zhivago (1...|   Drama|Romance|War|   2067|4.923286437988281|
+-------+--------------------+--------------------+-------+-----------------+



In [40]:
# toPandas(): Pandas 데이터프레임으로 출력
recs.toPandas()

,movieId,title,genres,movieID,rating
0,1209,Once Upon a Time in the West (C'era una volta ...,Action|Drama|Western,1209,5.039458
1,8827,"Bill Cosby, Himself (1983)",Comedy|Documentary,8827,5.014345
2,714,Dead Man (1995),Drama|Mystery|Western,714,5.005677
3,3470,Dersu Uzala (1975),Adventure|Drama,3470,4.968241
4,2067,Doctor Zhivago (1965),Drama|Romance|War,2067,4.923286
